# Evaluasi Tengah Semester INF-1316 Organisasi dan Arsitektur Komputer
Dosen: Mulya Agung
## Simulator Data Hazards CPU dan Virtual Memory

**Petunjuk**
1. Isi `NIM` pada sel berikut, lalu jalankan semua sel dari atas (*Run All*).
2. **Jangan mengubah** sel generator soal dan sel konstanta. Parameter soal berbeda untuk setiap NIM.
3. Lengkapi setiap fungsi bertanda `# TODO` pada soal A-D dan **hapus baris `raise NotImplementedError`**. Jangan mengganti nama atau parameter fungsi.
4. Kode akan diuji otomatis dengan program grader menggunakan NIM Anda **dan** lima NIM lain, jadi fungsi harus bekerja untuk parameter apa pun.
5. Kumpulkan notebook ini ke eCampuz dengan nama `UTS_INF1316_<NIM>.ipynb`.

In [ ]:
NIM = "ISI_NIM_ANDA"

## Generator soal (jangan diubah)

In [ ]:
# JANGAN DIUBAH: generator parameter soal per NIM
import random

PAGE_SIZE = 4096
SKELETONS = [  # register abstrak a..g dipetakan ke R1..R7 secara acak per NIM
    [("I1", "LOAD",  "a", [],         "m0"),
     ("I2", "ADD",   "b", ["a", "c"], None),
     ("I3", "SUB",   "a", ["d", "e"], None),
     ("I4", "STORE", None, ["b"],     "m1"),
     ("I5", "ADD",   "b", ["f", "g"], None),
     ("I6", "LOAD",  "c", [],         "m2"),
     ("I7", "STORE", None, ["c"],     "m3")],
    [("I1", "LOAD",  "a", [],         "m0"),
     ("I2", "SUB",   "b", ["c", "d"], None),
     ("I3", "ADD",   "c", ["a", "b"], None),
     ("I4", "LOAD",  "a", [],         "m1"),
     ("I5", "STORE", None, ["c"],     "m2"),
     ("I6", "ADD",   "e", ["a", "f"], None),
     ("I7", "STORE", None, ["e"],     "m3")],
    [("I1", "ADD",   "a", ["b", "c"], None),
     ("I2", "LOAD",  "b", [],         "m0"),
     ("I3", "STORE", None, ["a"],     "m1"),
     ("I4", "SUB",   "a", ["b", "d"], None),
     ("I5", "LOAD",  "c", [],         "m2"),
     ("I6", "ADD",   "e", ["a", "c"], None),
     ("I7", "STORE", None, ["e"],     "m3")],
]

def _outcome(program, pt):
    """Hitung jumlah page fault & protection fault (mode user) untuk validasi."""
    faults = prot = 0
    for _, op, _, _, addr in program:
        if addr is None: continue
        pte = pt[addr // PAGE_SIZE]
        w = op == "STORE"
        if pte["sup"] or (w and not pte["write"]) or (not w and not pte["read"]):
            prot += 1
        elif not pte["present"]:
            faults += 1
    return faults, prot

def generate(nim):
    rng = random.Random(int("".join(ch for ch in str(nim) if ch.isdigit())))
    while True:
        skeleton = rng.choice(SKELETONS)
        regs = rng.sample([f"R{i}" for i in range(1, 8)], 7)
        rmap = dict(zip("abcdefg", regs))
        vpns = rng.sample(range(4), 4)              # VPN untuk m0..m3
        addr = {f"m{k}": vpns[k] * PAGE_SIZE + rng.randrange(0, PAGE_SIZE, 4) for k in range(4)}
        program = [(n, op, rmap.get(d), [rmap[s] for s in src], addr.get(m))
                   for n, op, d, src, m in skeleton]
        frames = rng.sample(range(1, 16), 6)
        absent = set(rng.sample(range(4), 2))
        pt = {}
        for v in range(4):
            pt[v] = {"present": v not in absent, "ppn": None if v in absent else frames[v],
                     "read": 1, "write": rng.randint(0, 1), "sup": int(rng.random() < 0.25)}
        free = frames[4:6]
        f, p = _outcome(program, pt)
        if f >= 1 and p >= 1:                        # pastikan ada page fault & protection fault
            return program, pt, free

In [ ]:
PAGE_FAULT_PENALTY = 100  # cycle

class PageFault(Exception): pass
class ProtectionFault(Exception): pass

PROGRAM, PAGE_TABLE, FREE_FRAMES = generate(NIM)

## Parameter soal Anda

In [ ]:
print("Program:")
for name, op, dest, srcs, addr in PROGRAM:
    print(f"  {name}: {op:<5} tujuan={dest or '-':<4} sumber={', '.join(srcs) or '-':<8} alamat={hex(addr) if addr is not None else '-'}")
print("\nPage table (page size 4 KB, mode user):")
for vpn, e in PAGE_TABLE.items():
    print(f"  VPN {vpn}: present={int(e['present'])} ppn={e['ppn']} READ={e['read']} WRITE={e['write']} SUP={e['sup']}")
print("\nFREE_FRAMES =", FREE_FRAMES)

## A. Deteksi data hazards (24 poin)
Kembalikan list tuple `(jenis, instr_awal, instr_akhir, register)` untuk **semua** pasangan (i, j) dengan i < j:
- **RAW**: register tujuan Ii dibaca Ij
- **WAR**: register sumber Ii ditulis Ij
- **WAW**: Ii dan Ij menulis register yang sama

In [ ]:
def detect_hazards(program):
    # TODO
    raise NotImplementedError

for h in detect_hazards(PROGRAM):
    print(h)

## B. Register renaming (18 poin)
Ganti setiap register tujuan dengan register fisik baru `P0, P1, ...` sesuai urutan, dan perbarui register sumber pada instruksi berikutnya mengikuti pemetaan terbaru.

In [ ]:
def rename_registers(program):
    # TODO
    raise NotImplementedError

renamed = rename_registers(PROGRAM)
for ins in renamed:
    print(ins)
print("Hazard tersisa:", detect_hazards(renamed))

## C. Virtual memory (34 poin)
**`translate`** (24 poin): hitung VPN = `vaddr // PAGE_SIZE` dan offset. Periksa berurutan:
1. `SUP = 1` dan bukan supervisor → `ProtectionFault`
2. akses `"read"` dengan `READ = 0`, atau `"write"` dengan `WRITE = 0` → `ProtectionFault`
3. `present = False` → `PageFault(vpn)`

Kembalikan alamat fisik = `PPN * PAGE_SIZE + offset`.

**`handle_page_fault`** (10 poin): ambil frame kosong pertama dari `free_frames`, tulis sebagai PPN, set `present = True`.

In [ ]:
def translate(vaddr, access, page_table, supervisor=False):
    # TODO
    raise NotImplementedError

def handle_page_fault(vpn, page_table, free_frames):
    # TODO
    raise NotImplementedError

for name, op, _, _, addr in PROGRAM:
    if addr is None:
        continue
    access = "read" if op == "LOAD" else "write"
    try:
        print(name, hex(addr), "->", hex(translate(addr, access, PAGE_TABLE)))
    except PageFault as e:
        print(name, hex(addr), "-> PageFault pada VPN", e.args[0])
    except ProtectionFault as e:
        print(name, hex(addr), "-> ProtectionFault:", e)

## D. Eksekusi dan kinerja (24 poin)
**`raw_stalls`** (8 poin): stall setiap pasangan RAW = `max(0, 3 - jarak)`, dengan jarak = selisih indeks instruksi.

**`run`** (16 poin): jalankan program, kembalikan `(total_cycles, jumlah_page_fault, total_stall, log)`.
- 1 cycle per instruksi (termasuk yang dibatalkan), +100 cycle per page fault, ditambah total stall RAW.
- Setelah page fault: tangani, lalu **ulangi** instruksi.
- Protection fault: catat di log, instruksi dibatalkan, lanjut ke instruksi berikutnya.
- **Salin** `page_table` dan `free_frames` sebelum diubah; jangan mengubah milik pemanggil.

In [ ]:
def raw_stalls(program):
    # TODO
    raise NotImplementedError

def run(program, page_table, free_frames, supervisor=False):
    # TODO
    raise NotImplementedError

cycles, faults, stalls, log = run(PROGRAM, PAGE_TABLE, FREE_FRAMES)
for entry in log:
    print(entry)
print(f"\npage fault = {faults}, stall RAW = {stalls}, total cycle = {cycles}")